<a href="https://colab.research.google.com/github/jrodriguezrosales/Procesamiento-de-imagenes-en-Python/blob/main/segmentacion_imagenes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Segmentación de Imágenes con OpenCV
### Serie: Visión Artificial — Análisis y Procesamiento Avanzado
**Autor: José Rodríguez Rosales**

---

| # | Técnica | ¿Cuándo usarla? |
|---|---------|------------------|
| 1 | **Umbralizado (Thresholding)** | Objetos con diferente brillo que el fondo |
| 2 | **Watershed** | Objetos que se tocan o se solapan |
| 3 | **Detección de Contornos** | Extraer y analizar los bordes de regiones |

> 💡 **¿Qué es la segmentación?** Dividir una imagen en regiones significativas: separar monedas del fondo, aislar células, detectar vehículos.

**Instrucciones:** Ejecuta las celdas en orden. Las celdas 🎯 son ejercicios interactivos.

## ⚙️ Paso 0 — Configuración del entorno


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from google.colab.patches import cv2_imshow
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas correctamente')
print(f'   OpenCV versión: {cv2.__version__}')
print(f'   NumPy versión:  {np.__version__}')

## 🖼️ Paso 1 — Cargar la imagen

Dos opciones: imagen sintética de ejemplo o tu propia foto.

In [ ]:
def mostrar(imagenes, titulos, cmap_list=None, figsize=(16, 5)):
    """Muestra varias imágenes en una fila con sus títulos."""
    n = len(imagenes)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1: axes = [axes]
    for i, (img_i, titulo) in enumerate(zip(imagenes, titulos)):
        cmap = None
        if cmap_list: cmap = cmap_list[i]
        elif len(img_i.shape) == 2: cmap = 'gray'
        img_show = cv2.cvtColor(img_i, cv2.COLOR_BGR2RGB) if len(img_i.shape)==3 else img_i
        axes[i].imshow(img_show, cmap=cmap)
        axes[i].set_title(titulo, fontsize=13, fontweight='bold')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

print('✅ Función mostrar() definida')

In [ ]:
def crear_imagen_monedas(n=10, seed=42):
    """Imagen sintética: monedas doradas sobre fondo gris oscuro."""
    np.random.seed(seed)
    fondo = np.random.randint(30, 70, (400, 600, 3), dtype=np.uint8)
    img_s = fondo.copy()
    radios  = np.random.randint(30, 65, n)
    centros = []
    colores = [(200,180,100),(210,190,110),(190,170,90),(205,185,105),(215,195,115)]
    for i in range(n):
        intentos = 0
        while intentos < 200:
            cx = np.random.randint(radios[i]+5, 595-radios[i])
            cy = np.random.randint(radios[i]+5, 395-radios[i])
            solapado = any(np.sqrt((cx-px)**2+(cy-py)**2)<(radios[i]+pr-5) for px,py,pr in centros)
            if not solapado: break
            intentos += 1
        centros.append((cx, cy, radios[i]))
        color = colores[i % len(colores)]
        for r in range(radios[i], 0, -1):
            factor = r / radios[i]
            c = tuple(int(x*(0.6+0.4*factor)) for x in color)
            cv2.circle(img_s, (cx,cy), r, c, 1)
        cv2.circle(img_s, (cx,cy), radios[i], color, 2)
    grad = np.tile(np.linspace(0.7, 1.0, 600), (400,1))
    img_s = (img_s * grad[:,:,None]).clip(0,255).astype(np.uint8)
    return img_s

img_ejemplo = crear_imagen_monedas(n=10)
print('🪙 Imagen sintética creada (monedas claras sobre fondo oscuro)')
mostrar([img_ejemplo], ['Imagen de ejemplo — Monedas sintéticas'], figsize=(8,5))

In [ ]:
# ─── Opción B: Sube tu propia imagen ────────────────────────────────────
# Descomenta las siguientes líneas para subir tu foto:
# uploaded = files.upload()
nombre = '/content/monedas.jpg'
img_propia = cv2.imread(nombre)
# mostrar([img_propia], [f'Tu imagen: {nombre}'], figsize=(8,5))

# ─── Selección activa ────────────────────────────────────────────────────
# Cambia img_ejemplo por img_propia si subiste tu foto
img  = img_propia.copy()
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print(f'📐 Resolución: {img.shape[1]}×{img.shape[0]} px  |  Canales: {img.shape[2]}')
mostrar([img, gray], ['Original (color)', 'Escala de grises'])

---
## 📊 Análisis previo — Histograma y detección automática del tipo de imagen


In [ ]:
# — Histograma y detección automática por área
val_otsu, thresh_otsu_tmp = cv2.threshold(gray, 0, 255,
                                          cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Área de la clase blanca (255) y clase negra (0)
area_blanca = np.sum(thresh_otsu_tmp == 255)
area_negra  = np.sum(thresh_otsu_tmp == 0)

# La clase minoritaria es la que representa los objetos
# Si la clase blanca es minoritaria → objetos claros; si es mayoritaria → objetos oscuros
objetos_claros = area_blanca < area_negra

# Initialize media_fondo and media_objeto to prevent NameError in unexpected scenarios
media_objeto = None
media_fondo = None

# Calcular media_fondo y media_objeto
if objetos_claros:
    # Si los objetos son claros (blancos en thresh_otsu_tmp), su media es la de los píxeles blancos
    media_objeto = np.mean(gray[thresh_otsu_tmp == 255])
    media_fondo  = np.mean(gray[thresh_otsu_tmp == 0])
else:
    # Si los objetos son oscuros (negros en thresh_otsu_tmp), su media es la de los píxeles negros
    media_objeto = np.mean(gray[thresh_otsu_tmp == 0])
    media_fondo  = np.mean(gray[thresh_otsu_tmp == 255])

print(f'📌 Umbral Otsu: {int(val_otsu)}')
print(f'   Media brillo fondo  : {media_fondo:.1f}')
print(f'   Media brillo objetos: {media_objeto:.1f}')
if objetos_claros:
    print('   ✅ Modo detectado: OBJETOS CLAROS sobre fondo oscuro (ej: coins.jpg)')
    print('      → Watershed usará THRESH_BINARY_INV para que objetos queden en blanco')
else:
    print('   ✅ Modo detectado: OBJETOS OSCUROS sobre fondo claro')
    print('      → Watershed usará THRESH_BINARY directo')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Imagen en escala de grises', fontweight='bold')
axes[0].axis('off')
hist = cv2.calcHist([gray],[0],None,[256],[0,256]).flatten()
axes[1].fill_between(range(256), hist, alpha=0.6, color='steelblue')
axes[1].plot(range(256), hist, color='navy', linewidth=0.8)
axes[1].axvline(val_otsu, color='crimson', lw=2, linestyle='--', label=f'Umbral Otsu={int(val_otsu)}')
axes[1].axvline(127, color='orange', lw=2, linestyle=':', label='Umbral manual=127')
zona = 'Objetos claros ☀️' if objetos_claros else 'Objetos oscuros 🌑'
axes[1].set_title(f'Histograma — {zona}', fontweight='bold')
axes[1].set_xlabel('Intensidad (0=negro, 255=blanco)')
axes[1].set_ylabel('Píxeles')
axes[1].legend()
axes[1].set_xlim([0,255])
plt.tight_layout()
plt.show()

---
## 1️⃣ Método 1: Umbralizado (Thresholding)

```
píxel > umbral  →  255 (blanco = objeto)
píxel ≤ umbral  →    0 (negro  = fondo)
```

OpenCV ofrece **tres variantes** con creciente inteligencia:

In [ ]:
# ─── 1.1 Umbral binario simple ──────────────────────────────────────────
_, thresh_bin = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
print('📌 Umbral BINARIO SIMPLE  •  t=127  •  Sensible a iluminación no uniforme')
mostrar([gray, thresh_bin], ['Original en grises', 'Umbral binario (t=127)'])

In [ ]:
# ─── 1.2 Umbral adaptativo ──────────────────────────────────────────────
thresh_adapt = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
)
print('📌 Umbral ADAPTATIVO  •  blockSize=11  •  Robusto ante luz no uniforme ✅')
mostrar([gray, thresh_bin, thresh_adapt], ['Original','Umbral fijo t=127','Adaptativo'])

In [ ]:
# ─── 1.3 Umbral de Otsu ─────────────────────────────────────────────────
val_otsu, thresh_otsu = cv2.threshold(gray, 0, 255,
                                       cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f'📌 Umbral OTSU  •  t={int(val_otsu)} (automático)  •  Sin parámetros manuales ✅')
mostrar([thresh_bin, thresh_adapt, thresh_otsu],
        [f'Binario (t=127)', 'Adaptativo', f'Otsu (t={int(val_otsu)} auto)'])

### 🎯 Ejercicio interactivo 1 — Explora el efecto del umbral

Mueve el slider y observa cómo cambia la binarización en tiempo real.

In [ ]:
slider = widgets.IntSlider(
    value=int(val_otsu), min=0, max=255, step=1,
    description='Umbral:', style={'description_width':'initial'},
    layout=widgets.Layout(width='60%')
)
output_t = widgets.Output()

def actualizar_t(change):
    with output_t:
        clear_output(wait=True)
        t = change['new']
        _, th = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
        fig, axes = plt.subplots(1, 3, figsize=(16,4))
        axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
        etiq = '⬅ Demasiado bajo' if t<int(val_otsu)*0.5 else ('Demasiado alto ➡' if t>200 else '✅ Rango útil')
        axes[1].imshow(th, cmap='gray')
        axes[1].set_title(f'Umbral = {t}  {etiq}', fontweight='bold'); axes[1].axis('off')
        hist = cv2.calcHist([gray],[0],None,[256],[0,256]).flatten()
        axes[2].fill_between(range(256), hist, alpha=0.5, color='steelblue')
        axes[2].axvline(t, color='crimson', lw=2, label=f'Manual={t}')
        axes[2].axvline(int(val_otsu), color='green', lw=2, ls='--', label=f'Otsu={int(val_otsu)}')
        axes[2].legend(); axes[2].set_title('Histograma'); axes[2].set_xlim([0,255])
        plt.tight_layout(); plt.show()

slider.observe(actualizar_t, names='value')
display(slider, output_t)
actualizar_t({'new': int(val_otsu)})

### 📋 Comparativa: los tres métodos

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18,8))
variantes = [
    (gray,        'Original\n(grises)',            'gray',   '—'),
    (thresh_bin,  f'Binario\nt=127',               'gray', 'cv2.THRESH_BINARY'),
    (thresh_adapt,'Adaptativo\nblockSize=11',      'gray', 'ADAPTIVE_THRESH_GAUSSIAN_C'),
    (thresh_otsu, f'Otsu\nt={int(val_otsu)} auto', 'gray', 'THRESH_BINARY+THRESH_OTSU'),
]
pros   = ['—','Muy rápido','Ilum. irregular','Sin params manuales']
contras = ['—','Sensible a luz','Más lento','Necesita hist. bimodal']
for i,(im,titulo,cmap,codigo) in enumerate(variantes):
    axes[0][i].imshow(im, cmap=cmap); axes[0][i].set_title(titulo,fontsize=11,fontweight='bold'); axes[0][i].axis('off')
    axes[1][i].axis('off')
    axes[1][i].text(0.5,0.5,f'Código:\n{codigo}\n\n✅ {pros[i]}\n❌ {contras[i]}',
                    ha='center',va='center',fontsize=9,transform=axes[1][i].transAxes,
                    bbox=dict(boxstyle='round',facecolor='lightyellow',alpha=0.8))
plt.suptitle('Comparativa de métodos de umbralizado',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.show()

---
## 2️⃣ Método 2: Algoritmo Watershed

### ¿Por qué no basta el umbralizado?

Cuando los objetos **se tocan o se solapan**, el umbral los ve como un único blob. Watershed los separa.

**Analogía:** imagina que la imagen es un mapa de relieve. Watershed la inunda desde los valles. Cuando dos masas de agua de cuencas distintas se encuentran, construye una pared — el borde entre objetos.

---
> La binarización de Watershed debe producir siempre **objetos en BLANCO sobre fondo NEGRO**.
> Esto depende del tipo de imagen:
> - **Objetos claros** (coins.jpg con fondo negro): usar `THRESH_BINARY_INV` → invierte para que monedas queden blancas
> - **Objetos oscuros** (fondo claro): usar `THRESH_BINARY` directamente

> El notebook lo **detecta automáticamente** gracias a la variable `objetos_claros` calculada antes.

### Pipeline Watershed — 6 pasos:

In [ ]:
img_ws  = img.copy()
gray_ws = gray.copy()
print('🔄 Imagen recargada para el pipeline Watershed')

In [ ]:
# PASO 1: Binarización adaptada al tipo de imagen
# ────────────────────────────────────────────────────────────────────────
# REGLA: la imagen binaria resultante debe tener OBJETOS=BLANCO, FONDO=NEGRO
#
# Si los objetos son CLAROS (ej: monedas plateadas/doradas sobre fondo negro):
#   THRESH_BINARY_INV  -> invierte la imagen binaria estándar
#   (el umbral normal dejaría objetos en blanco, pero Otsu a veces los pone en negro)
#   Ajustamos según 'objetos_claros' calculado anteriormente.
#
# Si los objetos son OSCUROS (ej: texto negro sobre papel blanco):
#   THRESH_BINARY directo

if objetos_claros:
    # Objetos claros (mayor brillo) -> sin invertir, ya quedan blancos
    _, binary = cv2.threshold(gray_ws, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    print('PASO 1 ✅  Modo: OBJETOS CLAROS -> aplicando THRESH_BINARY + OTSU')
else:
    # Objetos oscuros (menor brillo) -> invertimos para que los objetos sean blancos
    _, binary = cv2.threshold(gray_ws, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    print('PASO 1 ✅  Modo: OBJETOS OSCUROS -> aplicando THRESH_BINARY_INV + OTSU')

# Verificar: en la imagen binaria los objetos deben ser BLANCOS
pct_blanco = np.sum(binary==255) / binary.size * 100
print(f'         Píxeles blancos (objetos): {pct_blanco:.1f}%  '
      f'| Píxeles negros (fondo): {100-pct_blanco:.1f}%')
if pct_blanco > 60:
    print('         ⚠️  Más del 60% es blanco -> el fondo y objeto podrían estar invertidos.')
    print('            Considera invertir manualmente: binary = cv2.bitwise_not(binary)')

mostrar([gray_ws, binary], ['Grises', 'Binaria (objetos=blanco, fondo=negro)'])

In [ ]:
# PASO 2: Limpieza morfológica – cierre (rellena huecos) + apertura (elimina ruido)
kernel_close = np.ones((3, 3), np.uint8)   # tamaño suficiente para cerrar números/grabados
kernel_open  = np.ones((3, 3), np.uint8)

# 1) Cierre: dilata y luego erosiona → rellena agujeros negros dentro de los blobs blancos
closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close, iterations=1)

# 2) Apertura: elimina pequeños puntos blancos fuera de los objetos
opening = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel_open, iterations=1)

print('PASO 2 ➤ Cierre (rellenar huecos) + Apertura (quitar ruido)')
mostrar([binary, closed, opening],
        ['Binaria original', 'Tras cierre (sin huecos)', 'Tras apertura (final)'])

In [ ]:
# PASO 3: Transformada de distancia
# Cada píxel blanco recibe valor = distancia al borde negro más cercano
# Los centros de los objetos tienen los valores más altos (= picos)
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
dist_norm = cv2.normalize(dist_transform, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

print('PASO 3 ✅  Transformada de distancia (DIST_L2)')
print('           Picos brillantes = centros de los objetos')
fig, axes = plt.subplots(1,3,figsize=(16,5))
for ax,im,t,cm in zip(axes,
    [opening, dist_norm, dist_norm],
    ['Apertura morfológica','Transformada de distancia\n(mapa de calor hot)',
     'Mapa viridis\n(amarillo = centro de objeto)'],
    ['gray','hot','viridis']):
    ax.imshow(im, cmap=cm); ax.set_title(t, fontweight='bold'); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# PASO 4: Definir regiones seguras
# sure_fg : píxeles que con CERTEZA son objeto (centros)
# sure_bg : píxeles que con CERTEZA son fondo
# unknown : zona de duda donde Watershed decidirá el borde
_, sure_fg = cv2.threshold(dist_transform, 0.5 * dist_transform.max(), 255, 0)
sure_fg    = np.uint8(sure_fg)
sure_bg    = cv2.dilate(opening, kernel_open, iterations=3)
unknown    = cv2.subtract(sure_bg, sure_fg)

print('PASO 4 ✅  Regiones seguras definidas')
print(f'   Fondo seguro  (sure_bg): {np.sum(sure_bg==255):,} px')
print(f'   Objeto seguro (sure_fg): {np.sum(sure_fg==255):,} px')
print(f'   Zona dudosa   (unknown): {np.sum(unknown==255):,} px')

fig, axes = plt.subplots(1,3,figsize=(16,5))
for ax,(im,t,cm) in zip(axes,
    [(sure_bg,'Fondo seguro (sure_bg)','Greys'),
     (sure_fg,'Objeto seguro: centros (sure_fg)','Greys'),
     (unknown,'Zona desconocida\n(bordes a resolver)','Reds')]):
    ax.imshow(im,cmap=cm); ax.set_title(t,fontweight='bold'); ax.axis('off')
plt.suptitle('PASO 4: Regiones seguras',fontsize=14); plt.tight_layout(); plt.show()

In [ ]:
# PASO 5: Marcadores para Watershed
n_labels, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1          # fondo = 1 (no 0)
markers[unknown == 255] = 0    # zona dudosa = 0 (Watershed decidirá aquí)
print(f'PASO 5 ✅  {n_labels-1} semillas (objetos) detectadas')
markers_vis = cv2.applyColorMap(
    (markers * (255 // max(markers.max(),1))).astype(np.uint8), cv2.COLORMAP_JET)
mostrar([markers_vis],[f'Mapa de marcadores — {n_labels-1} semillas'])

In [ ]:
# PASO 6: Aplicar Watershed
# markers == -1  =>  borde entre dos objetos distintos
img_ws_result = img_ws.copy()
markers_ws    = cv2.watershed(img_ws_result, markers.copy())
img_ws_result[markers_ws == -1] = [255, 0, 0]   # bordes en AZUL (visible sobre fondo negro)

# Versión coloreada: un color por objeto
img_coloreada = np.zeros_like(img_ws)
np.random.seed(7)
for label in np.unique(markers_ws):
    if label <= 1: continue
    color = [int(x) for x in np.random.randint(60,230,3)]
    img_coloreada[markers_ws == label] = color
img_coloreada[markers_ws == -1] = [255,255,255]  # bordes blancos

n_objetos = len([l for l in np.unique(markers_ws) if l > 1])
print(f'PASO 6 ✅  Watershed completado — {n_objetos} objetos separados')

fig, axes = plt.subplots(1,3,figsize=(18,5))
for ax,im,t in zip(axes,
    [img_ws, img_ws_result, img_coloreada],
    ['Original','Watershed — bordes azules',f'Cada objeto coloreado ({n_objetos} detectados)']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(t, fontsize=12, fontweight='bold'); ax.axis('off')
plt.suptitle('PASO 6: Resultado final del Watershed',fontsize=14)
plt.tight_layout(); plt.show()

### 🎯 Ejercicio interactivo 2 — Ajusta el factor de sure_fg

El `factor` controla qué tan conservador es el centro seguro (sure_fg):

- **Factor bajo (0.1):** más área como 'objeto seguro' → más semillas → puede partir objetos en dos
- **Factor alto (0.9):** solo el pico central → menos semillas → puede perder objetos pequeños

**Objetivo:** encuentra el factor que detecte exactamente el número correcto de objetos.

In [ ]:
slider_ws = widgets.FloatSlider(
    value=0.5, min=0.05, max=0.95, step=0.05,
    description='Factor:', readout_format='.2f',
    style={'description_width':'initial'}, layout=widgets.Layout(width='60%')
)
output_ws = widgets.Output()

def actualizar_ws(change):
    with output_ws:
        clear_output(wait=True)
        factor = change['new']
        _, sfg = cv2.threshold(dist_transform, factor*dist_transform.max(), 255, 0)
        sfg = np.uint8(sfg)
        sbg = cv2.dilate(opening, kernel_open, iterations=3)
        unk = cv2.subtract(sbg, sfg)
        nl, mrk = cv2.connectedComponents(sfg)
        mrk = mrk+1; mrk[unk==255] = 0
        img_tmp = img_ws.copy()
        mrk_ws = cv2.watershed(img_tmp, mrk.copy())
        img_tmp[mrk_ws==-1] = [255,0,0]
        n_obj = len([l for l in np.unique(mrk_ws) if l>1])
        fig, axes = plt.subplots(1,2,figsize=(14,5))
        axes[0].imshow(cv2.cvtColor(img_ws, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original'); axes[0].axis('off')
        axes[1].imshow(cv2.cvtColor(img_tmp, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'Factor={factor:.2f}  →  {n_obj} objetos detectados', fontweight='bold')
        axes[1].axis('off')
        plt.tight_layout(); plt.show()

slider_ws.observe(actualizar_ws, names='value')
display(slider_ws, output_ws)
actualizar_ws({'new': 0.5})

### 🗺️ Resumen visual del pipeline Watershed

In [ ]:
pasos = [
    (binary,       'PASO 1\nBinarización\n(objetos=blanco)'),
    (opening,      'PASO 2\nApertura\nmorfológica'),
    (dist_norm,    'PASO 3\nTransformada\nde distancia'),
    (unknown,      'PASO 4\nZona\ndesconocida'),
    (markers_vis,  'PASO 5\nMarcadores\n(semillas)'),
    (img_ws_result,'PASO 6\nWatershed\nfinal'),
]
fig, axes = plt.subplots(1,6,figsize=(22,4))
for ax,(im,titulo) in zip(axes,pasos):
    if len(im.shape)==3: ax.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB))
    else: ax.imshow(im,cmap='gray')
    ax.set_title(titulo,fontsize=9,fontweight='bold'); ax.axis('off')
plt.suptitle('Pipeline completo del algoritmo Watershed',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()

---
## 3️⃣ Método 3: Detección de Contornos

Una vez segmentada la imagen binaria, extraemos los **contornos** para medir, clasificar y contar objetos.

**Usos:** 📏 medir áreas · 🔷 clasificar formas · 🎯 bounding boxes · 📊 contar objetos

In [ ]:
# RETR_EXTERNAL : solo contornos externos (sin huecos interiores)
# CHAIN_APPROX_SIMPLE : comprime segmentos rectos, guarda solo vértices
# Asegurarnos de tener una imagen binaria con objetos en blanco (255)
if objetos_claros:
    # Los objetos ya son brillantes -> sin invertir, quedan blancos
    _, binary_ct = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
else:
    # Objetos oscuros -> hay que invertir para que se vuelvan blancos
    _, binary_ct = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

contours, _ = cv2.findContours(binary_ct, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f' 🔍 Contornos encontrados: {len(contours)}')
img_contornos = img.copy()
cv2.drawContours(img_contornos, contours, -1, (0,255,0), 2)
mostrar([img, thresh_otsu, img_contornos],
        ['Original','Binaria (Otsu)',f'Todos los contornos ({len(contours)})'],
        figsize=(18,5))

In [ ]:
# Filtrar por área mínima (elimina ruido de contornos pequeños)
area_min = 500
contornos_filtrados = [cnt for cnt in contours if cv2.contourArea(cnt) > area_min]
print(f'Total: {len(contours)}  →  Filtrados (area > {area_min} px²): {len(contornos_filtrados)}')
img_filtrada = img.copy()
cv2.drawContours(img_filtrada, contornos_filtrados, -1, (0,255,0), 2)
mostrar([img_contornos, img_filtrada],
        [f'Todos ({len(contours)})', f'Filtrados area>{area_min} px² ({len(contornos_filtrados)})'],
        figsize=(14,5))

In [ ]:
# Análisis de propiedades por contorno
img_anotada = img.copy()
datos = []
for i, cnt in enumerate(contornos_filtrados):
    area      = cv2.contourArea(cnt)
    perimetro = cv2.arcLength(cnt, True)
    circ      = (4 * np.pi * area) / (perimetro**2 + 1e-6)
    (cx,cy), radio = cv2.minEnclosingCircle(cnt)
    datos.append({'id':i+1,'area':area,'perimetro':perimetro,
                  'circularidad':circ,'radio':radio})
    cv2.drawContours(img_anotada,[cnt],-1,(0,255,0),2)
    cv2.circle(img_anotada,(int(cx),int(cy)),int(radio),(255,100,0),1)
    cv2.putText(img_anotada,str(i+1),(int(cx)-8,int(cy)+5),
                cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,0,255),2)

print(f'{'ID':>4} {'Area':>10} {'Perimetro':>12} {'Circular.':>12} {'Radio':>8}')
print('-'*52)
for d in datos:
    print(f"{d['id']:>4} {d['area']:>10.1f} {d['perimetro']:>12.1f} "
          f"{d['circularidad']:>12.3f} {d['radio']:>8.1f}")
mostrar([img_anotada],[f'{len(contornos_filtrados)} objetos anotados'],figsize=(9,6))

In [ ]:
# Estadísticas visuales
areas  = [d['area'] for d in datos]
circs  = [d['circularidad'] for d in datos]
radios = [d['radio'] for d in datos]
ids    = [f"#{d['id']}" for d in datos]

fig, axes = plt.subplots(1,3,figsize=(18,4))
axes[0].bar(ids, areas, color='steelblue', edgecolor='navy')
axes[0].axhline(np.mean(areas), color='crimson', ls='--', label=f'Media:{np.mean(areas):.0f}')
axes[0].set_title('Área por objeto (px²)', fontweight='bold'); axes[0].legend()

colors = ['green' if c>0.7 else 'orange' if c>0.4 else 'red' for c in circs]
axes[1].bar(ids, circs, color=colors, edgecolor='black')
axes[1].axhline(0.7, color='green', ls='--', alpha=0.5)
axes[1].set_title('Circularidad (1.0=círculo perfecto)', fontweight='bold')
axes[1].set_ylim([0,1.2])
ley = [mpatches.Patch(color='green',label='>0.7 Circular'),
       mpatches.Patch(color='orange',label='0.4–0.7'),
       mpatches.Patch(color='red',label='<0.4 Irregular')]
axes[1].legend(handles=ley, fontsize=9)

sc = axes[2].scatter(areas, radios, c=circs, cmap='RdYlGn', s=100, edgecolors='black')
for a,r,label in zip(areas,radios,ids):
    axes[2].annotate(label,(a,r),xytext=(5,5),textcoords='offset points',fontsize=9)
axes[2].set_title('Área vs. Radio (color=circularidad)', fontweight='bold')
plt.colorbar(sc, ax=axes[2], label='Circularidad')
plt.suptitle('Análisis estadístico de los contornos',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()

### 🎯 Ejercicio interactivo 3 — Filtro de circularidad

**Verde** = objeto circular (pasa el filtro) · **Rojo** = descartado por irregular

In [ ]:
slider_circ = widgets.FloatSlider(
    value=0.6, min=0.0, max=1.0, step=0.05,
    description='Circularidad min:', readout_format='.2f',
    style={'description_width':'initial'}, layout=widgets.Layout(width='60%')
)
output_circ = widgets.Output()

def actualizar_circ(change):
    with output_circ:
        clear_output(wait=True)
        circ_min = change['new']
        img_tmp = img.copy()
        sel = 0
        for cnt in contornos_filtrados:
            a = cv2.contourArea(cnt); p = cv2.arcLength(cnt,True)
            c = (4*np.pi*a)/(p**2+1e-6)
            color = (0,255,0) if c>=circ_min else (0,0,255)
            grosor = 3 if c>=circ_min else 1
            if c>=circ_min: sel+=1
            cv2.drawContours(img_tmp,[cnt],-1,color,grosor)
        fig,axes = plt.subplots(1,2,figsize=(14,5))
        axes[0].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original'); axes[0].axis('off')
        axes[1].imshow(cv2.cvtColor(img_tmp,cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'Circ≥{circ_min:.2f}  →  {sel} objeto(s) verde(s)',fontweight='bold')
        axes[1].axis('off'); plt.tight_layout(); plt.show()

slider_circ.observe(actualizar_circ,names='value')
display(slider_circ,output_circ)
actualizar_circ({'new':0.6})

---
## 🔄 Pipeline completo reutilizable

Función que encadena los tres métodos y se adapta automáticamente al tipo de imagen:

```
Original → Otsu adaptativo → Watershed → Contornos → Métricas
```

In [ ]:
def segmentar_y_analizar(
        imagen_bgr,
        area_minima=500,
        factor_ws=0.5,
        kernel_close_size=11,
        kernel_close_iter=3,
        usar_hough=False,
        hough_param1=100,
        hough_param2=20,
        hough_minDist=40,
        hough_minRadius=20,
        hough_maxRadius=150,
        verbose=True,
):
    """
    Pipeline Watershed (versión 5), con una corrección importante.

    Corregido un bug clave (presente en v4 y anteriores):
    ────────────────────────────────────────────────────
    Antes, `findContours` se aplicaba sobre la imagen `opening` (que es
    binaria y *pre-watershed*). Esto significaba que, aunque el Watershed
    separaba bien las regiones, el conteo de objetos se basaba en los blobs
    originales, a menudo fusionados, lo cual no reflejaba el resultado real
    del Watershed.

    La solución en esta v5:
    ──────────────────────
    Ahora, después de aplicar Watershed, iteramos sobre cada etiqueta única
    en `mrk_ws`. Para cada etiqueta (que representa un objeto separado por el
    Watershed), creamos una máscara binaria y luego extraemos su contorno.
    Así, el conteo de objetos es *exacto* a lo que Watershed ha segmentado.

    Otros detalles y mejoras:
    ────────────────────────
    - Se aplica `MORPH_CLOSE` en *ambos* modos (Hough y distancia)
    - La detección de objetos claros/oscuros se basa en el área (no solo el brillo)
    - Se añade un `GaussianBlur` antes de la binarización para mejorar la robustez
    """
    img_b  = imagen_bgr.copy()
    gray_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)

    # ### Paso 0: Suavizado (Blur)
    gray_blur = cv2.GaussianBlur(gray_b, (5, 5), 0)

    # ### Paso 1: Binarización adaptativa por área
    val_otsu, tmp = cv2.threshold(gray_blur, 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    obj_claros = np.sum(tmp == 255) < np.sum(tmp == 0)
    tipo = (cv2.THRESH_BINARY     + cv2.THRESH_OTSU) if obj_claros else \
           (cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    _, bin_b = cv2.threshold(gray_blur, 0, 255, tipo)

    # ### Paso 2a: Apertura — elimina ruido exterior
    k3 = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(bin_b, cv2.MORPH_OPEN, k3, iterations=2)

    # ### Paso 2b: Cierre — rellena huecos de textura
    # Se aplica en *ambos* modos (Hough y distancia). Antes solo se usaba
    # en modo distancia, dejando los blobs con huecos en modo Hough.
    k_big   = np.ones((kernel_close_size, kernel_close_size), np.uint8)
    closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, k_big,
                                iterations=kernel_close_iter)

    # ### Pasos 3-5: Semillas para Watershed
    if usar_hough:
        mrk, n_sem = _semillas_hough(
            gray_b, closing, k3,
            hough_param1, hough_param2, hough_minDist,
            hough_minRadius, hough_maxRadius,
        )
        dt_viz = sfg_viz = None
    else:
        mrk, n_sem, dt_viz, sfg_viz = _semillas_distancia(
            closing, k3, factor_ws
        )

    # ### Paso 6: Aplicar Watershed
    mrk_ws = cv2.watershed(img_b, mrk)
    img_b[mrk_ws == -1] = [255, 80, 0]   # Bordes resaltados en naranja

    # ### Paso 7: Extracción de Contornos desde los labels de Watershed (¡la clave de esta versión!)
    cnts_f = []
    datos  = []
    labels_validos = [l for l in np.unique(mrk_ws) if l > 1]  # Excluir 0 (desconocido) y 1 (fondo)

    for label in sorted(labels_validos):
        # Crear una máscara binaria para esta región específica
        mask = np.zeros(mrk_ws.shape, dtype=np.uint8)
        mask[mrk_ws == label] = 255

        # Encontrar los contornos de esta máscara
        region_cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)
        if not region_cnts:
            continue
        cnt = max(region_cnts, key=cv2.contourArea) # Tomar el contorno más grande si hay varios

        area = cv2.contourArea(cnt)
        if area < area_minima:
            continue

        perim = cv2.arcLength(cnt, True)
        circ  = (4 * np.pi * area) / (perim ** 2 + 1e-6)
        (cx, cy), radio = cv2.minEnclosingCircle(cnt)
        i = len(datos) + 1 # Generar ID secuencial para los objetos válidos

        datos.append({'id': i, 'area': area, 'perimetro': perim,
                      'circularidad': circ, 'radio': radio})
        cnts_f.append(cnt)

        cv2.drawContours(img_b, [cnt], -1, (0, 255, 0), 2) # Dibujar contorno en verde
        cv2.putText(img_b, str(i), (int(cx) - 8, int(cy) + 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 0), 2) # Número de objeto en amarillo

    if verbose:
        _mostrar(imagen_bgr, img_b, opening, closing, dt_viz, sfg_viz,
                 usar_hough, n_sem, len(datos),
                 'CLAROS' if obj_claros else 'OSCUROS', int(val_otsu), datos)

    return {'imagen': img_b, 'contornos': cnts_f, 'datos': datos}


# --- Funciones auxiliares para el pipeline (helpers) ---

def _semillas_distancia(closing, k3, factor_ws):
    """Genera semillas usando la transformada de distancia."""
    dt = cv2.distanceTransform(closing, cv2.DIST_L2, 5)
    dt_viz = dt.copy()

    _, sfg = cv2.threshold(dt, factor_ws * dt.max(), 255, 0)
    sfg = np.uint8(sfg)
    sfg_viz = sfg.copy()

    sbg = cv2.dilate(closing, k3, iterations=3)
    unk = cv2.subtract(sbg, sfg)

    _, mrk = cv2.connectedComponents(sfg)
    n_sem = int(mrk.max())
    mrk   = mrk + 1
    mrk[unk == 255] = 0

    return mrk, n_sem, dt_viz, sfg_viz


def _semillas_hough(gray_b, closing, k3,
                    p1, p2, minDist, minR, maxR):
    """Genera semillas detectando círculos con la transformada de Hough."""
    blur    = cv2.medianBlur(gray_b, 5)
    circles = cv2.HoughCircles(
        blur, cv2.HOUGH_GRADIENT, dp=1.2,
        minDist=minDist, param1=p1, param2=p2,
        minRadius=minR, maxRadius=maxR,
    )

    mrk   = np.zeros(closing.shape, dtype=np.int32)
    n_sem = 0

    if circles is not None:
        circles = np.uint16(np.around(circles))
        for i, (cx, cy, r) in enumerate(circles[0], start=2):
            seed_r = max(5, int(r * 0.2)) # Tamaño de semilla relativo al radio del círculo
            cv2.circle(mrk, (cx, cy), seed_r, i, -1)
            n_sem += 1

    sure_bg = cv2.dilate(closing, k3, iterations=3)
    mrk[sure_bg == 0] = 1 # Marcar el fondo seguro como 1

    return mrk, n_sem


def _mostrar(orig, resultado, opening, closing,
             dt_viz, sfg_viz, usar_hough, n_sem, n_obj, modo, val_otsu, datos_obj):
    """Muestra los pasos intermedios y el resultado final del pipeline."""
    if usar_hough or dt_viz is None:
        fig, axes = plt.subplots(1, 4, figsize=(22, 5))
        imgs = [orig, opening, closing, resultado]
        tits = [
            'Original',
            'Apertura (con huecos)',
            'Cierre (rellena huecos)',
            f'Resultado v5: {n_obj} objetos ({n_sem} semillas)',
        ]
        highlight = 2
    else:
        fig, axes = plt.subplots(1, 5, figsize=(26, 4))
        dt_show = cv2.normalize(dt_viz, None, 0, 255,
                                cv2.NORM_MINMAX).astype(np.uint8)
        imgs = [orig, opening, closing, dt_show, resultado]
        tits = [
            'Original',
            'Apertura (con huecos)',
            'Cierre (rellena huecos)',
            'Transformada de Distancia sobre closing',
            f'Resultado v5: {n_obj} objetos',
        ]
        highlight = 2

    for i, (ax, im, tit) in enumerate(zip(axes, imgs, tits)):
        show = cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if len(im.shape) == 3 else im
        ax.imshow(show, cmap='gray' if len(im.shape) == 2 else None)
        color = 'tab:red' if i == highlight else 'black'
        ax.set_title(tit, fontweight='bold' if i == highlight else 'normal',
                     color=color)
        ax.axis('off')

    plt.suptitle(
        f'Pipeline Watershed v5 | Objetos {modo} | Umbral Otsu={val_otsu} | '
        f'Contornos extraídos de los labels de Watershed',
        fontsize=12, fontweight='bold',
    )
    plt.tight_layout()
    plt.show()

    print(f'\nModo detectado: {modo} | Umbral Otsu: {val_otsu} | Objetos totales: {n_obj}')
    print(f"{'ID':>4} {'Área':>10} {'Perímetro':>12} {'Circ.':>8} {'Radio':>8}")
    print('─' * 48)
    for d in datos_obj:
        print(f"{d['id']:>4} {d['area']:>10.1f} {d['perimetro']:>12.1f} "
              f"{d['circularidad']:>8.3f} {d['radio']:>8.1f}")

---
## 📚 Cuadro comparativo final

In [ ]:
import pandas as pd
resumen = {
    'Metodo': ['Umbral simple','Umbral adaptativo','Umbral Otsu','Watershed','Contornos'],
    'Velocidad': ['Muy rapido','Rapido','Rapido','Moderado','Rapido'],
    'Objetos solapados': ['No','No','No','Si','Parcial'],
    'Parametros': ['Umbral t','blockSize, C','Ninguno ✅','factor','area_min'],
    'Imagen oscura/clara': ['Manual','Auto','Auto','Auto detecta ✅','Depende previo'],
    'Funcion OpenCV': ['threshold','adaptiveThreshold','THRESH_OTSU','watershed','findContours'],
}
df = pd.DataFrame(resumen).set_index('Metodo')
print('CUADRO COMPARATIVO')
print('='*95)
print(df.to_string())
print('\nRegla practica:')
print('  1. Objetos con brillo diferente al fondo -> Otsu (detecta umbral solo)')
print('  2. Iluminacion variable                  -> Umbral adaptativo')
print('  3. Objetos solapados/tocandose           -> Watershed (con deteccion auto)')
print('  4. Medir / clasificar / contar           -> Contornos (siempre ultimo paso)')

---
## 🎯 Desafío final: ¡Prueba con tu propia imagen!

El pipeline detecta automáticamente si tus objetos son claros u oscuros.
Funciona con monedas, frutas, células, texto, o cualquier conjunto de objetos.

In [ ]:
print('Sube tu imagen (jpg, png):')
uploaded = files.upload()
if uploaded:
    nombre = list(uploaded.keys())[0]
    img_propia = cv2.imread(nombre)
    if img_propia is None:
        print('No se pudo leer la imagen. Verifica el formato.')
    else:
        h, w = img_propia.shape[:2]
        if max(h, w) > 900:
            scale = 900 / max(h, w)
            img_propia = cv2.resize(img_propia, (int(w*scale), int(h*scale)))
        print(f'Ejecutando pipeline en: {nombre}  |  {img_propia.shape[1]}×{img_propia.shape[0]} px')
        resultado = segmentar_y_analizar(
            img_propia,
            area_minima=1000,
            usar_hough=True,          # Hough para monedas que se tocan
            hough_param2=20,
            hough_minDist=50,
            hough_minRadius=20,
            hough_maxRadius=100,
            kernel_close_size=7,
            kernel_close_iter=7,
        )
else:
    print('No se subió imagen. Usa el ejemplo de las celdas anteriores.')

---
## 🏁 Cierre y próximos pasos

### Lo que aprendiste hoy:

| ✅ | Concepto |
|----|----------|
| ✅ | Qué es la segmentación y por qué importa |
| ✅ | Umbralizado simple, adaptativo y Otsu |
| ✅ | Detección automática de tipo de imagen (objetos claros/oscuros) |
| ✅ | Pipeline Watershed completo en 6 pasos |
| ✅ | Detección, filtrado y análisis de contornos |
| ✅ | Métricas: área, perímetro, circularidad |
| ✅ | Pipeline reutilizable adaptativo |

### Próximo video: **Detección de Características — ORB, SIFT y SURF**

---
> 👍 Si te gustó el tutorial, deja un like y suscríbete al canal.
>
> 💬 Cualquier duda, déjala en los comentarios. ¡Nos vemos en el siguiente código!

**Referencias:**
- [OpenCV Docs — Thresholding](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html)
- [OpenCV Docs — Watershed](https://docs.opencv.org/4.x/d3/db4/tutorial_py_watershed.html)
- [OpenCV Docs — Contours](https://docs.opencv.org/4.x/d4/d73/tutorial_py_contours_begin.html)